[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Testing and Packaging](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)

# Parametrize &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell sets up the project's folders and `run_pytest`, and the cell after it writes the
module as the notebook left it. Run them first, then the tasks in order, since tasks 2 to 4 rewrite
the file task 1 writes. The last cell removes the scratch folder.


In [1]:
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT = Path("scratch/stations")
(PROJECT / "tests").mkdir(parents=True, exist_ok=True)
os.environ["NO_COLOR"] = "1"                  # programs started from here print without color codes
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"   # and keep no compiled copies, which a quick rewrite can outrun


def pytest_report(*arguments, folder=PROJECT):
    """What python -m pytest prints when it runs in the folder, less what differs between computers."""
    settings = {"COLUMNS": "80", "PYTEST_DISABLE_PLUGIN_AUTOLOAD": "1", "PYTHONNODEBUGRANGES": "1"}
    finished = subprocess.run([sys.executable, "-m", "pytest", "--no-header", *arguments],
                              cwd=folder, capture_output=True, text=True, env={**os.environ, **settings})
    report = finished.stdout + finished.stderr
    report = report.replace(f"{Path(folder).resolve()}/", "")          # the folder's own path
    report = re.sub(r"\S*/_pytest/", "_pytest/", report)                # the path to pytest's own files
    return re.sub(r" in \d+\.\d+s\b", "", report).rstrip()              # the time the run took


def run_pytest(*arguments, folder=PROJECT):
    """Run python -m pytest in the folder, as a terminal would, and print its report."""
    print(pytest_report(*arguments, folder=folder))


print("ready:", PROJECT)


ready: scratch/stations


In [2]:
%%writefile scratch/stations/readings.py
"""Readings from the weather stations, and each station's mean temperature."""

import statistics


def parse_reading(line):
    """A (station, celsius) pair from a line such as 'Bergen,4.2'. An empty reading is None.

    A minus sign written as U+2212, as some spreadsheets write it, reads as a hyphen-minus.
    """
    station, celsius = line.strip().split(",")
    celsius = celsius.replace("\u2212", "-")
    return station, float(celsius) if celsius else None


def mean(values):
    """The mean of the readings that are not None, or None when there are none."""
    present = [value for value in values if value is not None]
    return statistics.fmean(present) if present else None


def summarize(lines):
    """Each station's mean temperature, from lines of readings. A blank line is skipped."""
    by_station = {}
    for line in lines:
        if not line.strip():
            continue
        station, celsius = parse_reading(line)
        by_station.setdefault(station, []).append(celsius)
    return {station: mean(values) for station, values in by_station.items()}


def summarize_file(path):
    """Each station's mean temperature, from a file of readings."""
    with open(path, encoding="utf-8-sig") as file:
        return summarize(file)


def to_fahrenheit(celsius):
    """A temperature in degrees Celsius, in degrees Fahrenheit."""
    return celsius * 9 / 5 + 32


Writing scratch/stations/readings.py


**1.** A parametrized test of mean.


In [3]:
%%writefile scratch/stations/tests/test_tasks.py
import pytest

from readings import mean


@pytest.mark.parametrize("values, expected", [([4.2, 5.8], 5.0), ([None, -6.3], -6.3), ([None], None)])
def test_mean(values, expected):
    assert mean(values) == expected


Writing scratch/stations/tests/test_tasks.py


In [4]:
run_pytest("tests/test_tasks.py", "-v")


============================= test session starts ==============================
collecting ... collected 3 items

tests/test_tasks.py::test_mean[values0-5.0] PASSED                       [ 33%]
tests/test_tasks.py::test_mean[values1--6.3] PASSED                      [ 66%]
tests/test_tasks.py::test_mean[values2-None] PASSED                      [100%]

============================== 3 passed ===============================


The lists became `values0` to `values2`, and the expected values, which are numbers and `None`, kept
their own text.


**2.** IDs with ids.


In [5]:
%%writefile scratch/stations/tests/test_tasks.py
import pytest

from readings import mean


@pytest.mark.parametrize("values, expected", [([4.2, 5.8], 5.0), ([None, -6.3], -6.3), ([None], None)],
                         ids=["two readings", "a missing reading", "no readings"])
def test_mean(values, expected):
    assert mean(values) == expected


Overwriting scratch/stations/tests/test_tasks.py


In [6]:
run_pytest("tests/test_tasks.py", "-v")


============================= test session starts ==============================
collecting ... collected 3 items

tests/test_tasks.py::test_mean[two readings] PASSED                      [ 33%]
tests/test_tasks.py::test_mean[a missing reading] PASSED                 [ 66%]
tests/test_tasks.py::test_mean[no readings] PASSED                       [100%]

============================== 3 passed ===============================


The IDs follow the order of the cases, which is what a list of IDs relies on.


**3.** One case, by its node ID.


In [7]:
run_pytest("tests/test_tasks.py::test_mean[no readings]", "-v")


============================= test session starts ==============================
collecting ... collected 1 item

tests/test_tasks.py::test_mean[no readings] PASSED                       [100%]

============================== 1 passed ===============================


The space inside the brackets is part of the ID. In a terminal, quote the whole node ID, since the
shell would otherwise split it at the space.


**4.** pytest.param, and an xfail.


In [8]:
%%writefile scratch/stations/tests/test_tasks.py
import pytest

from readings import mean


@pytest.mark.parametrize("values, expected", [
    pytest.param([4.2, 5.8], 5.0, id="two readings"),
    pytest.param([None, -6.3], -6.3, id="a missing reading"),
    pytest.param([None], None, id="no readings"),
    pytest.param(["4.2"], 4.2, id="a reading as text",
                 marks=pytest.mark.xfail(reason="readings arrive as numbers")),
])
def test_mean(values, expected):
    assert mean(values) == expected


Overwriting scratch/stations/tests/test_tasks.py


In [9]:
run_pytest("tests/test_tasks.py", "-v")


============================= test session starts ==============================
collecting ... collected 4 items

tests/test_tasks.py::test_mean[two readings] PASSED                      [ 25%]
tests/test_tasks.py::test_mean[a missing reading] PASSED                 [ 50%]
tests/test_tasks.py::test_mean[no readings] PASSED                       [ 75%]
tests/test_tasks.py::test_mean[a reading as text] XFAIL (readings ar...) [100%]

========================= 3 passed, 1 xfailed =========================


`mean(["4.2"])` raised `TypeError`, since `statistics.fmean` does not take text, and an `xfail`
counts any failure, an exception included, as the failure it expected.


**5.** Two stacked decorators.


In [10]:
%%writefile scratch/stations/tests/test_combinations.py
import pytest

from readings import parse_reading


@pytest.mark.parametrize("station", ["Bergen", "Oslo"])
@pytest.mark.parametrize("reading", ["4.2", ""])
def test_the_station_comes_first(station, reading):
    assert parse_reading(f"{station},{reading}")[0] == station


Writing scratch/stations/tests/test_combinations.py


In [11]:
report = pytest_report("tests/test_combinations.py", "-q")

print(report)
print("tests:", report.splitlines()[-1])


....                                                                     [100%]
4 passed
tests: 4 passed


Two stations and two readings make four tests.


**6.** A fixture with two line endings.


In [12]:
%%writefile scratch/stations/tests/test_line_endings.py
import pytest

from readings import summarize_file


@pytest.fixture(params=["\n", "\r\n"], ids=["unix", "windows"])
def line_ending(request):
    return request.param


def test_oslo_summarizes_with_either_line_ending(line_ending, tmp_path):
    path = tmp_path / "oslo.csv"
    path.write_text(line_ending.join(["Oslo,-2.4", "Oslo,-1.6"]) + line_ending, encoding="utf-8", newline="")

    summary = summarize_file(path)

    assert summary == {"Oslo": -2.0}


Writing scratch/stations/tests/test_line_endings.py


In [13]:
run_pytest("tests/test_line_endings.py", "-v")


============================= test session starts ==============================
collecting ... collected 2 items

tests/test_line_endings.py::test_oslo_summarizes_with_either_line_ending[unix] PASSED [ 50%]
tests/test_line_endings.py::test_oslo_summarizes_with_either_line_ending[windows] PASSED [100%]

============================== 2 passed ===============================


`newline=""` writes the endings exactly as given. Reading in text mode turns both kinds into `"\n"`,
and `parse_reading` strips what is left, so both runs pass.

Last, remove the scratch folder:


In [14]:
shutil.rmtree("scratch")

print("scratch still there:", Path("scratch").exists())


scratch still there: False


---

&#8592; **Back to:** [Parametrize](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/testing-and-packaging/05-parametrize.ipynb)  &nbsp;&middot;&nbsp;  [Testing and Packaging Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/testing-and-packaging.html)
